In [3]:
# ===============================================================
# AIT-626 | MMU-RAG Final Project
# TF-IDF Clustering + Wikipedia Corpus + Multi-Hop RAG
# + FLAN-T5-base Generator + Confidence Estimator
# ===============================================================

!pip -q install wikipedia sentence-transformers transformers tqdm pandas numpy scikit-learn

import os
import json
import warnings
import random

import numpy as np
import pandas as pd
import torch
import wikipedia

from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [4]:
# -----------------------------------------------------------
# Step 1: Load T2T dataset
# -----------------------------------------------------------
INPUT_JSONL = "/content/t2t_test_participants.jsonl"

queries = []
query_ids = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        q = obj.get("query", "").strip()
        if not q:
            continue
        queries.append(q)
        query_ids.append(obj.get("id", f"q_{len(query_ids)}"))

print(f"Loaded {len(queries)} queries.")
for i in range(min(3, len(queries))):
    print(f"[{query_ids[i]}] {queries[i]}")


Loaded 299 queries.
[q_0] Explain to me like I’m 5 the theory one can color in each map with 4 colors
[q_1] How does sunburn show on black skin?
[q_2] Explain why some clouds look different


In [5]:
# -----------------------------------------------------------
# Step 2: TF-IDF Vectorizer + KMeans Clustering
# -----------------------------------------------------------
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)
tfidf_matrix = tfidf.fit_transform(queries)
vocab = np.array(tfidf.get_feature_names_out())

if len(queries) <= 10:
    n_clusters = 2
else:
    n_clusters = min(10, max(2, len(queries) // 40))

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)
cluster_labels = kmeans.fit_predict(tfidf_matrix)

def top_keywords_for_cluster(cluster_id, top_k=8):
    idx = np.where(cluster_labels == cluster_id)[0]
    if len(idx) == 0:
        return []
    sub = tfidf_matrix[idx]
    mean_scores = np.asarray(sub.mean(axis=0)).ravel()
    top_idx = mean_scores.argsort()[::-1][:top_k]
    return vocab[top_idx].tolist()

print("\nCluster overview:")
for cid in range(n_clusters):
    kws = top_keywords_for_cluster(cid, top_k=8)
    print(f"  Cluster {cid} -> {np.sum(cluster_labels == cid)} queries")
    print(f"    Keywords: {kws}")

# -----------------------------------------------------------
# Simple keyword extractor for queries
# -----------------------------------------------------------
STOP_WORDS = {
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
    'should', 'may', 'might', 'can', 'to', 'of', 'in', 'on', 'at', 'for',
    'with', 'by', 'from', 'as', 'and', 'or', 'but', 'if', 'that', 'this',
    'these', 'those', 'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me',
    'my', 'your', 'his', 'her', 'its', 'our', 'their', 'what', 'which',
    'who', 'where', 'when', 'why', 'how', 'like', 'explain', 'tell'
}

def query_keywords(q, max_terms=5):
    words = [w.strip(".,!?;:()[]\"'").lower() for w in q.split()]
    words = [w for w in words if w and w not in STOP_WORDS and len(w) > 2]
    # Keep order but remove duplicates
    seen = set()
    result = []
    for w in words:
        if w not in seen:
            seen.add(w)
            result.append(w)
        if len(result) >= max_terms:
            break
    # Fallback: if nothing remains, just use first 3 raw words
    if not result:
        result = words[:3] if words else q.split()[:3]
    return result


Cluster overview:
  Cluster 0 -> 3 queries
    Keywords: ['data', 'data breaches', 'breaches occur', 'breaches', 'column', 'column data', 'scrape web', 'scrape']
  Cluster 1 -> 35 queries
    Keywords: ['explain', 'bake', 'cake', 'like', 'example', 'bake cake', 'bag', 'bake egg']
  Cluster 2 -> 30 queries
    Keywords: ['does', 'affect', 'pollution', 'environment', 'pollution affect', 'affect environment', 'climate change', 'climate']
  Cluster 3 -> 8 queries
    Keywords: ['work', 'epidurals', 'vaccines', 'vaccines work', 'epidurals work', 'inverter', 'royalties work', 'royalties']
  Cluster 4 -> 23 queries
    Keywords: ['make', 'pizza', 'sushi', 'make sushi', 'tell make', 'tell', 'way make', 'campfire']
  Cluster 5 -> 194 queries
    Keywords: ['did', 'use', 'caused', 'increasing', 'change', 'holocaust', 'works', 'print']
  Cluster 6 -> 6 queries
    Keywords: ['earth', 'mars', 'mars earth', 'systems', 'systems interact', 'interact', 'earth systems', 'earth possible']


In [6]:
# -----------------------------------------------------------
# Step 3: Wikipedia Corpus Building
# -----------------------------------------------------------
corpus_texts = []
corpus_sources = []
corpus_clusters = []

max_pages_per_query = 2          # small to keep runtime manageable
max_chars_per_page = 3000
max_queries_for_corpus = min(120, len(queries))  # limit to avoid huge corpus

print("\nBuilding Wikipedia corpus (query-driven)...")
print(f"Using first {max_queries_for_corpus} queries to build corpus.")

for idx in range(max_queries_for_corpus):
    qtext = queries[idx]
    cid = int(cluster_labels[idx])
    kw = query_keywords(qtext, max_terms=5)
    search_query = " ".join(kw)
    print(f"Query {idx} (cluster {cid}) search query: {search_query}")

    try:
        titles = wikipedia.search(search_query)[:max_pages_per_query]
    except Exception as e:
        print(f"  Search failed for query {idx}: {e}")
        continue

    for title in titles:
        try:
            page = wikipedia.page(title, auto_suggest=False)
            text = page.content[:max_chars_per_page]
            url = page.url
            corpus_texts.append(text)
            corpus_sources.append(url)
            corpus_clusters.append(cid)
        except Exception:
            continue

print(f"\nTotal passages in corpus: {len(corpus_texts)}")
if len(corpus_texts) == 0:
    raise RuntimeError("No passages collected. Check Wikipedia access or search logic.")

for i in range(min(3, len(corpus_texts))):
    print(f"\nPassage {i} (cluster {corpus_clusters[i]}):")
    print(corpus_texts[i][:250].replace("\n", " ") + "...")
    print("Source:", corpus_sources[i])




Building Wikipedia corpus (query-driven)...
Using first 120 queries to build corpus.
Query 0 (cluster 1) search query: i’m theory one color each
Query 1 (cluster 2) search query: sunburn show black skin
Query 2 (cluster 1) search query: some clouds look different
Query 3 (cluster 1) search query: convolution five
Query 4 (cluster 5) search query: stomach constantly empty put aluminum
Query 5 (cluster 1) search query: cosinus sinus image it's possible
Query 6 (cluster 5) search query: fly plane
Query 7 (cluster 6) search query: big earth ant put perspective
Query 8 (cluster 1) search query: milk manufa toring process
Query 9 (cluster 5) search query: visualize team teams structure powerpoint
Query 10 (cluster 5) search query: best practice using iphone camera
Query 11 (cluster 5) search query: xred worm remove
Query 12 (cluster 1) search query: electrical wiring house works
Query 13 (cluster 4) search query: know make storm32-bgc rotate behyond
Query 14 (cluster 5) search query: sum ca

In [7]:
# -----------------------------------------------------------
# Step 4: Embedding model and corpus encoding
# -----------------------------------------------------------
embed_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("\nEncoding corpus passages...")
corpus_embeddings = embed_model.encode(
    corpus_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Encoding corpus passages...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [8]:
# -----------------------------------------------------------
# Step 5: Multi-hop retrieval (2 hops, cluster-aware)
# -----------------------------------------------------------
def multi_hop_retrieve(query, cluster_id, top_k_first=4, top_k_second=3):
    idx = [i for i, c in enumerate(corpus_clusters) if c == cluster_id]
    if not idx:
        idx = list(range(len(corpus_texts)))

    local_embeds = corpus_embeddings[idx]

    q_embed = embed_model.encode(query, convert_to_tensor=True)
    first_scores = util.cos_sim(q_embed, local_embeds)[0]
    k1 = min(top_k_first, len(idx))
    first_vals, first_idx_local = torch.topk(first_scores, k=k1)
    first_idx_global = [idx[i.item()] for i in first_idx_local]
    first_passages = [corpus_texts[i] for i in first_idx_global]
    best_passage = first_passages[0]

    refined_query = query + " " + best_passage[:300]
    refined_embed = embed_model.encode(refined_query, convert_to_tensor=True)
    second_scores = util.cos_sim(refined_embed, local_embeds)[0]
    k2 = min(top_k_second, len(idx))
    second_vals, second_idx_local = torch.topk(second_scores, k=k2)
    second_idx_global = [idx[i.item()] for i in second_idx_local]

    all_indices_global = []
    all_passages = []
    all_sources = []
    for gidx in first_idx_global + second_idx_global:
        if gidx not in all_indices_global:
            all_indices_global.append(gidx)
            all_passages.append(corpus_texts[gidx])
            all_sources.append(corpus_sources[gidx])

    return all_passages, all_sources


In [9]:
# -----------------------------------------------------------
# Step 6: Generator (FLAN-T5) and Confidence Estimator
# -----------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
generator = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

def generate_answer(query,
                    passages,
                    max_input_tokens=512,
                    max_output_tokens=200,
                    do_sample=False,
                    temperature=0.7,
                    top_p=0.9):
    context = " ".join(passages)
    prompt = (
        "You are a helpful tutor. Use the context to give a clear, detailed answer.\n\n"
        f"Question: {query}\n"
        f"Context: {context}\n"
        "Answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_input_tokens
    ).to(device)

    gen_kwargs = {"max_length": max_output_tokens}
    if do_sample:
        gen_kwargs.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": top_p
        })

    outputs = generator.generate(**inputs, **gen_kwargs)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.strip()

def confidence_estimator(query, main_answer, alt_answers=None):
    if alt_answers is None:
        alt_answers = []

    texts = [main_answer] + alt_answers
    embeddings = embed_model.encode(
        [query] + texts,
        convert_to_tensor=True
    )
    q_vec = embeddings[0]
    ans_vecs = embeddings[1:]

    q_main_sim = util.cos_sim(q_vec, ans_vecs[0])[0].item()

    self_sim = 0.0
    if len(ans_vecs) > 1:
        sims = util.cos_sim(ans_vecs, ans_vecs)
        n = sims.size(0)
        total = 0.0
        count = 0
        for i in range(n):
            for j in range(i + 1, n):
                total += sims[i, j].item()
                count += 1
        if count > 0:
            self_sim = total / count

    def normalize_cos(x):
        return max(0.0, min(1.0, (x + 1.0) / 2.0))

    q_main_norm = normalize_cos(q_main_sim)
    self_norm = normalize_cos(self_sim)
    confidence = 0.6 * q_main_norm + 0.4 * self_norm
    return float(confidence)

def answer_with_confidence(query, passages, num_samples=2):
    main_answer = generate_answer(query, passages, do_sample=False)
    alt_answers = []
    for _ in range(num_samples):
        alt = generate_answer(
            query,
            passages,
            do_sample=True,
            temperature=0.8,
            top_p=0.9
        )
        alt_answers.append(alt)
    conf = confidence_estimator(query, main_answer, alt_answers)
    return main_answer, conf, alt_answers


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
# -----------------------------------------------------------
# Step 7: Run pipeline on all queries
# -----------------------------------------------------------
print("\nRunning MMU-RAG pipeline over all queries...")

results = []
for qid, qtext, cid in tqdm(
    list(zip(query_ids, queries, cluster_labels)),
    total=len(queries),
    desc="Processing queries"
):
    passages, sources = multi_hop_retrieve(
        qtext,
        cluster_id=int(cid),
        top_k_first=4,
        top_k_second=3
    )

    answer, conf, alt_answers = answer_with_confidence(
        qtext,
        passages,
        num_samples=2
    )

    results.append({
        "id": qid,
        "query": qtext,
        "cluster_id": int(cid),
        "answer": answer,
        "confidence": round(conf, 3),
        "sources": list(dict.fromkeys(sources)),
        "alt_answers": alt_answers
    })

print("\nFinished all queries.")



Running MMU-RAG pipeline over all queries...


Processing queries: 100%|██████████| 299/299 [05:14<00:00,  1.05s/it]


Finished all queries.


In [11]:
# -----------------------------------------------------------
# Step 8: Save outputs and show sample results
# -----------------------------------------------------------
OUTPUT_JSONL = "mmu_rag_results.jsonl"
OUTPUT_CSV = "mmu_rag_summary.csv"

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for r in results:
        json.dump(r, f)
        f.write("\n")

summary_rows = []
for r in results:
    summary_rows.append({
        "id": r["id"],
        "query": r["query"],
        "cluster_id": r["cluster_id"],
        "answer": r["answer"],
        "confidence": r["confidence"],
        "num_sources": len(r["sources"])
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved detailed results to: {OUTPUT_JSONL}")
print(f"Saved summary table to:   {OUTPUT_CSV}")

print("\nSample outputs:")
for r in results[:5]:
    print("-" * 100)
    print(f"ID:        {r['id']}  (cluster {r['cluster_id']})")
    print(f"Query:     {r['query']}")
    print(f"Confidence:{r['confidence']:.3f}")
    print(f"Sources:   {r['sources'][:3]}")
    print(f"Answer:    {r['answer'][:400]}...")
    print()



Saved detailed results to: mmu_rag_results.jsonl
Saved summary table to:   mmu_rag_summary.csv

Sample outputs:
----------------------------------------------------------------------------------------------------
ID:        q_0  (cluster 1)
Query:     Explain to me like I’m 5 the theory one can color in each map with 4 colors
Confidence:0.855
Sources:   ['https://en.wikipedia.org/wiki/Graph_coloring', 'https://en.wikipedia.org/wiki/Human_skin_color', 'https://en.wikipedia.org/wiki/List_of_cloud_types']
Answer:    The nature of the coloring problem depends on the number of colors but not on what they are....

----------------------------------------------------------------------------------------------------
ID:        q_1  (cluster 2)
Query:     How does sunburn show on black skin?
Confidence:0.691
Sources:   ['https://en.wikipedia.org/wiki/Sunburn', 'https://en.wikipedia.org/wiki/Dark_skin', 'https://en.wikipedia.org/wiki/Neptune']
Answer:    erythema...

----------------------------

In [12]:
import json
import numpy as np

RESULTS_FILE = "/content/mmu_rag_results.jsonl"

answer_lengths = []

with open(RESULTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        record = json.loads(line)

        # Handle both possible key names safely
        ans = record.get("answer") or record.get("final_answer") or ""
        ans = ans.strip()

        if ans:
            words = ans.split()
            answer_lengths.append(len(words))

if answer_lengths:
    avg_len = np.mean(answer_lengths)
    print(f"Number of answers: {len(answer_lengths)}")
    print(f"Average answer length: {avg_len:.2f} words")
    print(f"Min length: {min(answer_lengths)} words")
    print(f"Max length: {max(answer_lengths)} words")
else:
    print("No answers found in the file.")



Number of answers: 296
Average answer length: 9.02 words
Min length: 1 words
Max length: 66 words
